In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [5]:
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = [f"{group}_{t}" for group in groups for t in range(8)]
columns.append("target")

df = pd.read_csv(
    "TomsHardware.data",
    header=None,
    names=columns
)

df["target_log"] = np.log1p(df["target"])
print(df.shape)
#df.head(20)

(28179, 98)


In [1]:

# 1. Підготовка target
df["target_log"] = np.log1p(df["target"])

# 2. Формуємо X і y
# Прибираємо original target, log target і ND_ фічі
X = df.drop(
    ["target", "target_log"] + [col for col in df.columns if col.startswith("ND_")],
    axis=1
)
y = df["target_log"]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Lasso Regression with alpha selection
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

lasso_model = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

# 6. Predictions in log-space
y_pred_log = lasso_model.predict(X_test_scaled)

# 7. Metrics in log-space
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log = mean_absolute_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

# 8. Optional: predictions in original scale
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

rmse_real = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae_real = mean_absolute_error(y_test_real, y_pred_real)
r2_real = r2_score(y_test_real, y_pred_real)

# 9. Results
print("Best alpha:", lasso_model.alpha_)
print("\nMetrics in log-scale:")
print(f"RMSE: {rmse_log:.4f}")
print(f"MAE:  {mae_log:.4f}")
print(f"R²:   {r2_log:.4f}")

print("\nMetrics in original scale:")
print(f"RMSE: {rmse_real:.4f}")
print(f"MAE:  {mae_real:.4f}")
print(f"R²:   {r2_real:.4f}")

# 10. Optional: how many features were kept
non_zero_coef = np.sum(lasso_model.coef_ != 0)
print(f"\nNumber of selected features: {non_zero_coef} out of {X.shape[1]}")

NameError: name 'np' is not defined